# 기존 5fps 고정 + 경합 후보 전용 25fps · 최대 40 epoch

기존 실행을 중지한 뒤, 이 노트를 GPU 런타임에서 **모두 실행**하세요. 기존 Drive 데이터 ZIP을 재사용합니다. 추가 데이터 업로드나 코드 수정은 필요 없습니다.

- **5fps 모델:** 기존 PathCRF 가중치와 캐시 예측을 고정해 재사용
- **후보 선별:** 5fps 불확실도·상대 선수 접근·예측 전환으로 경합 의심 구간 선택. 정답을 보고 후보를 선택하지 않음
- **25fps 모델:** 후보 구간만 학습. 앞뒤 문맥은 입력으로 보지만 후보 밖은 loss 제외
- **결합:** 후보만 25fps로 교체, 후보 밖과 D 마스크는 기존 5fps 결과 유지
- **최대 40 epoch, 검증 loss 5 epoch 연속 개선 없음 → 얼리스타핑, 최소 검증 loss 모델 평가**
- AMP FP16 ON / 총 배치 64 / 임베딩 128 / 헤드 4 / 블록 2 / 4초 100프레임 창
- 학습 5경기 / 검증 J03WQQ / 평가 J03WR9. 모델 2개는 H/A 모델이며 기존 D 모델은 별도 고정 자료

이전 하이브리드의 후보 75% + 일반 25% 학습에서 **후보만 학습하도록 변경**한 실험입니다. 경기의 정답 구간을 미리 알고 선택하는 방식은 아닙니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import os,sys,json,csv,zipfile,shutil,hashlib,base64,gzip,subprocess

drive.mount('/content/drive')
drive_root=Path('/content/drive/MyDrive/transition_final')
archive=drive_root/'transition_colab_data.zip'
choices=[Path('/content/transition_25fps_project'),Path('/content/transition_project'),Path('/content/transition_candidate25_project')]
project=next((p for p in choices if (p/'data/transition_final/source/J03WR9.parquet').is_file()),choices[-1])
project.mkdir(parents=True,exist_ok=True)
if not (project/'data/transition_final/source/J03WR9.parquet').is_file():
    if not archive.is_file():raise FileNotFoundError(f'기존 입력 ZIP 위치: {archive}')
    with zipfile.ZipFile(archive) as z:z.extractall(project)
os.chdir(project)
print('기존 데이터 재사용:',project)


In [ ]:
# 실행 코드 내장
embedded='...'
for name,source in json.loads(gzip.decompress(base64.b64decode(embedded))).items():
    target=project/'transition final'/name
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(source,encoding='utf-8')
print('후보 전용 25fps 코드 준비 완료')


In [ ]:
%pip -q install 'kloppy==3.18.0' pyarrow scipy pandas matplotlib tqdm scikit-learn torch-geometric


In [ ]:
RUN_NAME='dfl_5fps_candidate25_40ep_es5'
MAX_EPOCHS=40
EARLY_STOP_PATIENCE=5
MICROBATCH=64  # OOM이면 학습 시작 전 32로 변경. 총 배치 64 유지.
EPOCH_WINDOW_CAP=0  # 후보 창 전체; 일반 창은 사용하지 않음

import torch
if not torch.cuda.is_available():raise RuntimeError('런타임 유형을 GPU로 선택하세요.')
run_dir=drive_root/'runs'/RUN_NAME
run_dir.mkdir(parents=True,exist_ok=True)
settings={'experiment':'frozen_5fps_candidate_only_25fps','epochs':MAX_EPOCHS,
          'early_stop':EARLY_STOP_PATIENCE,'microbatch':MICROBATCH,'epoch_window_cap':EPOCH_WINDOW_CAP,
          'amp':True,'amp_dtype':'float16','checkpoint_selection':'minimum_candidate_validation_loss'}
settings_path=run_dir/'colab_settings.json'
if settings_path.exists() and json.loads(settings_path.read_text())!=settings:
    raise ValueError('이 RUN_NAME의 기존 설정과 다릅니다. 새 RUN_NAME을 사용하세요.')
settings_path.write_text(json.dumps(settings,ensure_ascii=False,indent=2))
with (project/'transition final/parameters_candidate25.csv').open(newline='',encoding='utf-8') as f:
    parameters=list(csv.DictReader(f))
changes={'max_epochs':str(MAX_EPOCHS),'early_stop':str(EARLY_STOP_PATIENCE),'epoch_window_cap':str(EPOCH_WINDOW_CAP)}
for row in parameters:
    if row['parameter'] in changes:row['value']=changes[row['parameter']]
active_config=project/'transition final/parameters_candidate25_active.csv'
for target in (active_config,run_dir/'parameter_descriptions.csv'):
    with target.open('w',newline='',encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=['parameter','value','status','description']);w.writeheader();w.writerows(parameters)
print('GPU:',torch.cuda.get_device_name(0),'| AMP: ON | 총 배치: 64')
print('최대',MAX_EPOCHS,'epoch / 얼리스탑 patience',EARLY_STOP_PATIENCE,'/ 후보만 학습')
print('결과 저장:',run_dir)


In [ ]:
# 기존 5fps 출력 캐시를 별도 경로로 연결합니다. 이전 실험 캐시는 덮어쓰지 않습니다.
frozen_cache=project/'data/transition_final/candidate25_frozen'
frozen_cache.mkdir(parents=True,exist_ok=True)
for match in ('J03WMX','J03WN1','J03WOH','J03WOY','J03WPY','J03WQQ'):
    source=project/'frozen_for_colab'/f'{match}_pred.npz'
    if not source.is_file():raise FileNotFoundError(source)
    shutil.copy2(source,frozen_cache/source.name)
# 이미 계산한 평가 경기의 5fps 예측을 내장해 추가 CPU 추론 시간을 줄입니다. 정답 라벨은 들어 있지 않습니다.
heldout_cache_b64='...'
heldout_bytes=base64.b64decode(heldout_cache_b64)
if hashlib.sha256(heldout_bytes).hexdigest()!='e60e896c61388f53a53d06fff15862898c09285a27c758fdd6e5b5086f2db8d3':raise ValueError('내장 캐시 해시 불일치')
(frozen_cache/'J03WR9_pred.npz').write_bytes(heldout_bytes)
del heldout_bytes,heldout_cache_b64
upstream=project/'pathcrf'
checkpoint=upstream/'saved/140/model/state_dict_best_acc.pt'
with checkpoint.open('rb') as f:actual=hashlib.file_digest(f,'sha256').hexdigest()
if actual!='55757d9dca9777d60356ba368dc315867ea1ae38e90c332e935b9c5be72d83c9':raise ValueError('기존 5fps 체크포인트가 다릅니다.')
with (project/'data/pathcrf_dfl/scene_two_models_v1/full/best.pt').open('rb') as f:actual_D=hashlib.file_digest(f,'sha256').hexdigest()
if actual_D!='04079f6f570a48decb30e2ea521e878872ac490f196de72f35eb2ebcfb19c4e9':raise ValueError('기존 D 체크포인트가 다릅니다.')
print('기존 5fps 가중치·예측 캐시 연결 완료: 새로 학습하지 않습니다.')


In [ ]:
options=['--config',str(active_config),'--device','cuda','--microbatch',str(MICROBATCH),
         '--upstream',str(upstream),'--frozen-cache',str(frozen_cache),'--output',str(run_dir)]
def execute(stage):
    command=[sys.executable,'-u','transition final/run_candidate25.py',stage]+options
    print('실행:',stage,flush=True)
    with (run_dir/f'{stage}.log').open('a',encoding='utf-8') as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'{stage} 실패: {run_dir/(stage+".log")}')


## 후보 전문 모델 학습 → 평가

학습 셀을 실행하면 25fps 모델만 학습합니다. 검증 loss는 후보 프레임의 H/A CE + 전환 CE + CRF NLL이며, 최소값의 `best.pt`로 평가합니다. 개선이 없는 3 epoch마다 lr을 절반으로 낮추고, 5 epoch 연속 개선이 없으면 멈춥니다.

매 epoch 후보 학습 창 수·배치 수·실측 시간·최대 예산 기준 남은 시간·얼리스탑 대기 횟수를 출력합니다. 별도 벤치마크 학습은 실행하지 않습니다.

`last.pt`에 모델·optimizer·AMP scaler·난수 상태가 저장됩니다. 같은 설정과 RUN_NAME으로 다시 실행하면 마지막 완료 epoch 이후부터 이어집니다. epoch 중간에 끊기면 그 epoch를 다시 수행합니다.


In [ ]:
execute('train')
execute('test')
import pandas as pd
display(pd.read_csv(run_dir/'holdout_comparison.csv'))
print('학습 종료 정보:',json.loads((run_dir/'training_completed.json').read_text()))
print('선택된 epoch:',json.loads((run_dir/'best_validation.json').read_text())['epoch'])
print('완료:',run_dir)


## 저장된 파일

Drive `transition_final/runs/dfl_5fps_candidate25_40ep_es5/`에 자동 저장합니다.

- `parameters.csv`, `parameter_descriptions.csv`, `colab_settings.json`: 실제 적용 설정
- `manifest.json`, `runtime_info.json`, `gate.json`: 분할·고정 모델 해시·GPU·후보 기준
- `training_history.csv`, `best_validation.json`, `training_completed.json`: loss·Spell F1·선택 epoch·얼리스탑 결과
- `best.pt`, `last.pt`: 최선 모델과 재개 체크포인트
- `holdout_comparison.csv`, `holdout.json`: 5fps 대 5fps+후보 전용 25fps 비교
- `estimated_D_primary_J03WR9_H1.csv`, `H2.csv`: 원본 프레임별 truth·base·refined·candidate

후보 밖 예측과 D 마스크의 일치를 코드에서 확인합니다. H=0/A=1/D=2, 정답 결측=-1입니다. 평가 J03WR9는 이미 확인한 홀드아웃이므로 추가 진단 비교로 해석합니다.
